# Create NER evaluation dataset

## Init

In [ ]:
# # Enable autoreload to automatically pick up changes in local modules
# %load_ext autoreload
# %autoreload 2

import torch
assert torch.cuda.is_available(), "GPU still not detected"

In [ ]:
"""
NER evaluation and dataset creation pipeline for Greek archaeological documents.
This notebook initializes the environment, loads GLiNER2, and provides utilities
for data annotation and Argilla dataset management.
"""

import ast
import base64
import difflib
import io
import itertools
import json
import logging
import pprint
import random
import sys
import traceback
import urllib.request
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from logging.config import dictConfig
from logging.handlers import RotatingFileHandler
from pathlib import Path
from typing import List, Optional, Union

# Third-party ML and Data libraries
import argilla as rg
import html2text
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import regex as re
import requests
import seaborn as sns
# import spacy
from bs4 import BeautifulSoup
from datasets import (
    Dataset,
    DatasetDict,
    DatasetInfo,
    Features,
    Sequence,
    Value,
    load_from_disk,
)
from dotenv import dotenv_values, find_dotenv
from gliner2 import GLiNER2
from gliner2.training.data import InputExample, TrainingDataset
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig
from huggingface_hub import DatasetCard, DatasetCardData
from tabulate import tabulate
from tqdm import tqdm
# from wtpsplit import SaT, WtP

# Project-specific imports
from archaeo_ner_greek.logging_config import LOGGING_CONFIG
from archaeo_ner_greek.utils import (
    configure_argilla_client,
    configure_argilla_resources,
    get_dataset_as_dataframe,
)

# --- Logging Initialization ---
# --- Silence Argilla and associated Network Noise ---
# Sets everything to WARNING or higher (hiding INFO and DEBUG logs)
logging.getLogger("argilla").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
# --- Silence Python Warnings ---
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="argilla")
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

# --- Environment & Configuration ---
# Locate the root directory based on the .env file marker
env_path = find_dotenv()

if env_path:
    BASE_DIR = Path(env_path).resolve().parent
    env_vars = dotenv_values(env_path)
else:
    # Robust fallback: identify root if .env is missing
    logger.warning("No .env file found. Defaulting to directory traversal.")
    BASE_DIR = Path.cwd().resolve()
    if BASE_DIR.name == "notebooks":
        BASE_DIR = BASE_DIR.parent
    env_vars = {}

PROJECT_NAME = BASE_DIR.name
DATA_DIR = BASE_DIR / "data"
LOG_DIR = Path.home() / "logs"

# Plotting setup
sns.set_theme()

# Ensure required artifact directories exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Project root identified: {BASE_DIR}")
logger.info(f"Initialized project: {PROJECT_NAME}")
logger.info(f"Loaded environment variables: {list(env_vars.keys())}")


# 1. Data Loading and Preprocessing
In this section, we load the raw archaeological documents (CSV/JSON) and prepare them for the NER pipeline. This includes text cleaning, context windowing, and splitting into train/test sets.


In [ ]:
DEFAULT_ANNOTATOR = env_vars.get("ANNOTATOR_A")
df_annotated = get_dataset_as_dataframe(
    client=configure_argilla_client(env_vars=env_vars),
    dataset_name=env_vars.get("ARGILLA_DATASET"),
    workspace_name=env_vars.get("ARGILLA_WORKSPACE"), 
    username=DEFAULT_ANNOTATOR
)
if not df_annotated.empty:
    logger.info(f"Ready: {len(df_annotated)} samples loaded with 'labels' ready for training.")
    # You can now access labels directly: df_annotated['labels']

### Inspect the DataFrame Structure


In [ ]:
# --- Inspect the DataFrame Structure ---
logger.info(f"DataFrame Shape: {df_annotated.shape}")
logger.info(f"Available Columns: {df_annotated.columns.tolist()}")
# --- Decisive Debug Cell ---
if not df_annotated.empty:
    import json
    row = df_annotated.iloc[0]
    logger.info(f"{'='*40} FULL ROW DEBUG {'='*40}")
    logger.info(f"ID     : {row['id']}")
    logger.info(f"Full Response Dict: {json.dumps(row['sentence_field'], indent=2, ensure_ascii=False)}")
    logger.info(f"Labels (Extracted): {row['labels']}")
    logger.debug(f"Full Response Dict: {json.dumps(row['response'], indent=2, ensure_ascii=False)}")

In [ ]:
# 1. New Guidelines Dictionary
guidelines_dict = {
    "ARTEFACT": "movable archaeological find or inscription in Greek. Examples: 'αγγείο', 'επιγραφή'",
    "PERIOD": "archaeological or historical period, or ancient period abbreviation in Greek. Examples: 'ἑλληνιστικὴ περίοδος' '6ο αἰ. π.Χ.'",
    "LOCATION": "named geographic places, excluding building names. Examples: 'Ρόδος', 'Θάσος'",
    "CONTEXT": "archaeological layer, structure in situ associated with finds. Examples: 'τάφοι', 'βασιλική'",
    "MATERIAL": "substance from which an object or structure is made. Examples: 'χρυσός', 'μάρμαρο'",
    "SPECIES": "animal, plant, or mythological creature. Examples: 'γρύπας', 'μοσχαράκι', 'κισσός'",
    "PERSON": "historical or mythological person. Examples: 'Κυβέλης', 'Διός'",
    "FEATURE": "artistic style, iconographic motif, decoration type, or construction technique. Examples: 'ερυθρόμορφο' 'δωρικός'"
}
guidelines_dict = {k.lower(): v for k, v in guidelines_dict.items()}


# 2. Transform to InputExamples
train_examples = []
for _, row in df_annotated.iterrows():
    text = row['sentence_field']
    labels = row.get('labels', [])
    
    entities = {}
    for label_obj in labels:
        # lbl = label_obj['label']    
        lbl = label_obj['label'].lower() 

        start = label_obj['start']
        end = label_obj['end']
        mention = text[start:end].strip()
        
        if lbl not in entities:
            entities[lbl] = []
        entities[lbl].append(mention)
    
    train_examples.append(InputExample(
        text=text,
        entities=entities,
        entity_descriptions=guidelines_dict
    ))

# logger.info(train_examples[0])

# Step 2: Create and validate dataset
train_dataset = TrainingDataset(train_examples)
train_dataset.validate(raise_on_error=False)
train_dataset.print_stats()
from pprint import pprint
print("Labelset and definitions:")
pprint(guidelines_dict)
pprint(train_dataset[0])

# 2. Model Training (GLiNER 2.0)
We configure and execute the GLiNER 2.0 training process. This utilizes the semantic guidelines established for archaeological entities (ARTEFACT, PERIOD, etc.) to fine-tune the model on domain-specific Greek texts.


In [ ]:
experiment_name = f"gliner2_archaeo_lora_{datetime.now().strftime('%Y%m%d_%H%M')}"

# 1. Output Directory 
output_dir = BASE_DIR / f"data/models/{experiment_name}"
output_dir.mkdir(parents=True, exist_ok=True)

# 2. LoRA Training Configuration
def compute_metrics(model, eval_dataset, threshold=0.1):
    tp, fp, fn = 0, 0, 0
    labels = list(guidelines_dict.keys())
    model.eval()
    
    dataset_items = eval_dataset.examples if hasattr(eval_dataset, 'examples') else eval_dataset
    
    for ex in dataset_items:
        # 1. Inference (returns dict with 'entities' key)
        output = model.extract_entities(ex.text, labels, threshold=threshold)
        
        # 2. Extract Predictions from the dict
        # Predictions are in: output['entities'][label] = [mentions]
        pred_spans = []
        pred_entities = output.get('entities', {})
        for lbl, texts in pred_entities.items():
            for t in texts:
                pred_spans.append((t, lbl))
        
        # 3. Ground Truth from InputExample
        gt_spans = []
        for lbl, texts in ex.entities.items():
            for t in texts:
                gt_spans.append((t, lbl))
                
        # 4. Standard Exact Match Logic
        current_gt = gt_spans.copy()
        for p in pred_spans:
            if p in current_gt:
                tp += 1
                current_gt.remove(p)
            else:
                fp += 1
        fn += len(current_gt)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    
    metrics = {"f1": f1, "precision": precision, "recall": recall}
    
    # This print will show up in your training console logs
    print(f"\n>>> INTERIM EVAL: {metrics}")

    return metrics

training_config = TrainingConfig(
    output_dir=str(output_dir),
    experiment_name=experiment_name,
    seed=42,
    
    # Hardware & Batching Stability (CRITICAL)
    batch_size=1,
    eval_batch_size=1,             # Prevents "tensor size mismatch" during evaluation
    gradient_accumulation_steps=4, # Simulates Effective Batch Size = 4
    fp16=True,                     # Half-precision for speed/memory
    
    # LoRA Architecture (Rank 4 for stability on small datasets)
    use_lora=True,
    lora_r=4,                     # Reduced from 16
    lora_alpha=8.0,              # Reduced from 32.0 (standard 2*r)
    lora_dropout=0.1,             # Regularization for small datasets
    lora_target_modules=["encoder"], # Focused target
    save_adapter_only=True,        # Saves ~10-30MB instead of 1.2GB per checkpoint


    # Optimization Profile
    num_epochs=10,
    task_lr=1e-4,                 # Primary learning rate for adapters/heads
    warmup_ratio=0.1,
    scheduler_type="cosine",       # Smooth decay for stable convergence
    weight_decay=0.01,


    # Checkpointing (Accuracy follows Loss)
    eval_strategy="epoch",
    save_best=True,
    metric_for_best="eval_loss", # Use Loss to drive selection
    greater_is_better=False,      # Lower is better
    # Or
    # metric_for_best="eval_f1",  # Use F1 to drive selection
    # greater_is_better=True,    # Greater is better
    save_total_limit=2,
    logging_steps=5,
       
    # Early Stopping for Convergence
    early_stopping=False, # Disabled for initial experiments
    early_stopping_patience=10, # Allow for initial loss volatility

    # Data Handling
    validate_data=True,
    # strict_validation=False   
)

# 1. 90/10 Training/Validation Split
train_split, val_split, _ = train_dataset.split(
    train_ratio=0.9, 
    val_ratio=0.1, 
    test_ratio=0.0, 
    shuffle=True, 
    seed=42
)

# 2. Trainer with evaluation set
# model = GLiNER2.from_pretrained("fastino/gliner2-large-v1")
model = GLiNER2.from_pretrained("fastino/gliner2-multi-v1")
trainer = GLiNER2Trainer(model, training_config, compute_metrics=compute_metrics)

results = trainer.train(
    train_data=train_split, 
    eval_data=val_split
)

print(f"Training completed!")
print(f"Best validation loss: {results['best_metric']:.4f}")
print(f"Total steps: {results['total_steps']}")
print(f"Training time: {results['total_time_seconds']/60:.1f} minutes")

print(experiment_name)
print(results)

In [ ]:
# 1. Load the original base model 
best_model = GLiNER2.from_pretrained("fastino/gliner2-multi-v1")
# 2. Add the LoRA
adapter_path = f"../data/models/{experiment_name}/best"
best_model.load_adapter(adapter_path)
# 3. Ready for inference
print("Adapter loaded.")

# 3. Evaluation and Performance Analysis
Quantitative and qualitative assessment of the model's performance.
- **Metrics**: Precision, Recall, and F1-score.
- **Error Analysis**: Review of overlapping spans and MISC classification suggestions.


In [ ]:
def evaluate_adapter(model, adapter_path, test_data, threshold=0.1):
    """
    Loads a specific LoRA adapter and evaluates its performance.
    """
    # 1. Load the specific weights
    print(f"Loading adapter from: {adapter_path}")
    model.load_adapter(adapter_path)
    
    # 2. Execute metric calculation
    results = compute_metrics(model, test_data, threshold=threshold)
    
    print(f"\n--- EVALUATION RESULTS ({adapter_path.split('/')[-1]}) threshold: {threshold} ---")
    print(f"F1 Score : {results['f1']}")
    print(f"Precision: {results['precision']}")
    print(f"Recall   : {results['recall']}")
    print(f"Counts   : TP={results['tp']}, FP={results['fp']}, FN={results['fn']}")
    
    return results

final_results = evaluate_adapter(best_model, adapter_path, val_split, threshold=0.1)


Loading adapter from: ../data/models/gliner2_archaeo_lora_20260408_1654/best
2026-04-08 17:41:47,717 [INFO] gliner2.training.lora: Unloading existing adapter before loading new one
2026-04-08 17:41:47,719 [INFO] gliner2.training.lora: Unloaded 72 LoRA layers
2026-04-08 17:41:47,729 [INFO] gliner2.training.lora: Loaded 144 LoRA tensors from ../data/models/gliner2_archaeo_lora_20260408_1654/best/adapter_weights.safetensors
2026-04-08 17:41:47,740 [INFO] gliner2.training.lora: Applied LoRA to 72 layers
2026-04-08 17:41:47,741 [INFO] gliner2.training.lora: Loaded LoRA adapter from ../data/models/gliner2_archaeo_lora_20260408_1654/best

--- EVALUATION RESULTS (best) threshold: 0.1 ---
F1 Score : 0.2929
Precision: 0.1934
Recall   : 0.6029
Counts   : TP=41, FP=171, FN=27
